# 4) Inside PyPSA — Exercises

**Goals:** take PyPSA's model apart and count what is in it, rebuild lesson 3's KKT price decomposition from PyPSA's own duals, add a CO₂ budget with `extra_functionality` and check it against PyPSA's built-in `GlobalConstraint`, and reproduce the result in Pyomo.

The setup cell holds the lecture's day (now with CO₂ emission factors), a `network()` builder and the lecture's Pyomo model with an optional CO₂ budget. Power in MW, energy in MWh, emissions in t, prices in €/MWh, carbon prices in €/t.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import numpy as np
import pandas as pd
import pyomo.environ as pyo
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
logging.getLogger("pyomo").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

T = 24
HOURS = np.arange(T)
LOAD = 1900 + 400 * np.sin(np.pi * (HOURS - 6) / 12).clip(0) + 250 * np.exp(-((HOURS - 19) / 2.0) ** 2)
FREE = 2200 * 0.8 * np.clip(np.sin(np.pi * (HOURS - 6) / 13), 0, None) + 400.0
PLANTS = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150], "ramp": [60, 300, 500],
                       "co2": [0.90, 0.35, 0.50]},                      # t CO2 per MWh of electricity
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))
BATTERY = {"p_nom": 300, "e_nom": 1200, "eta": 0.95}


def network(with_carriers=False):
    """The lecture's day as a PyPSA network; with_carriers adds CO2 carriers for a GlobalConstraint."""
    n = pypsa.Network()
    n.set_snapshots(HOURS)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD)
    if with_carriers:
        n.add("Carrier", "free", co2_emissions=0.0)
        for gen, row in PLANTS.iterrows():
            n.add("Carrier", gen, co2_emissions=row.co2)
    n.add("Generator", "free", bus="HR", p_nom=FREE.max(), p_max_pu=FREE / FREE.max(), marginal_cost=0,
          carrier="free" if with_carriers else "")
    for gen, row in PLANTS.iterrows():
        n.add("Generator", gen, bus="HR", p_nom=row.p_nom, marginal_cost=row.mc, carrier=gen if with_carriers else "",
              ramp_limit_up=row.ramp / row.p_nom, ramp_limit_down=row.ramp / row.p_nom)
    n.add("StorageUnit", "battery", bus="HR", p_nom=BATTERY["p_nom"], max_hours=BATTERY["e_nom"] / BATTERY["p_nom"],
          efficiency_store=BATTERY["eta"], efficiency_dispatch=BATTERY["eta"], cyclic_state_of_charge=True)
    return n


def pyomo_dispatch(co2_limit=None):
    """The lecture's Pyomo model, with an optional daily CO2 budget (t) for the challenge."""
    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, T - 1)
    m.G = pyo.Set(initialize=list(PLANTS.index))
    m.g = pyo.Var(m.G, m.T, bounds=lambda m, g, t: (0, PLANTS.loc[g, "p_nom"]))
    m.f = pyo.Var(m.T, bounds=lambda m, t: (0, FREE[t]))
    m.c = pyo.Var(m.T, bounds=(0, BATTERY["p_nom"]))
    m.d = pyo.Var(m.T, bounds=(0, BATTERY["p_nom"]))
    m.e = pyo.Var(m.T, bounds=(0, BATTERY["e_nom"]))
    m.balance = pyo.Constraint(m.T, rule=lambda m, t: sum(m.g[g, t] for g in m.G) + m.f[t] + m.d[t] - m.c[t] == LOAD[t])
    m.soc = pyo.Constraint(m.T, rule=lambda m, t:
                           m.e[t] == m.e[(t - 1) % T] + BATTERY["eta"] * m.c[t] - m.d[t] / BATTERY["eta"])
    m.ramp_up = pyo.Constraint(m.G, m.T, rule=lambda m, g, t:
                               pyo.Constraint.Skip if t == 0 else m.g[g, t] - m.g[g, t - 1] <= PLANTS.loc[g, "ramp"])
    m.ramp_down = pyo.Constraint(m.G, m.T, rule=lambda m, g, t:
                                 pyo.Constraint.Skip if t == 0 else m.g[g, t - 1] - m.g[g, t] <= PLANTS.loc[g, "ramp"])
    if co2_limit is not None:
        m.co2 = pyo.Constraint(expr=sum(PLANTS.loc[g, "co2"] * m.g[g, t] for g in m.G for t in m.T) <= co2_limit)
    m.cost = pyo.Objective(expr=sum(PLANTS.loc[g, "mc"] * m.g[g, t] for g in m.G for t in m.T))
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

### Warm-ups

**1. What is in the model?**

Write `model_inventory(n)` that builds (but does not solve) PyPSA's model and returns a dict with the number of variables, the number of constraints, and the number of constraint entries that only express variable **bounds** (PyPSA names those `…-fix-…`). Show that the variable count matches lesson 3's 168, and that constraints − bound constraints gives lesson 3's 186 constraints.

In [2]:
def model_inventory(n) -> dict:
    """Variables, constraints and bound-type constraints of PyPSA's model of n."""
    ### BEGIN SOLUTION
    m = n.optimize.create_model()
    bounds = sum(int((m.constraints[c].labels != -1).sum()) for c in m.constraints if "-fix-" in c)
    return {"variables": m.nvars, "constraints": m.ncons, "bound_constraints": bounds}
    ### END SOLUTION

In [3]:
inv = model_inventory(network())
assert inv["variables"] == 168, "same number of variables as lesson 3"
assert inv["constraints"] - inv["bound_constraints"] == 186, "the rest are lesson 3's 186 constraints"
inv

{'variables': 168, 'constraints': 522, 'bound_constraints': 336}

**2. Lesson 3's price decomposition from PyPSA's own duals**

Solve the network with `assign_all_duals=True`, then read coal's ramp duals from `n.model` and rebuild the 16:00 price with lesson 3's formula λₜ = mc + (μ⁺ₜ − μ⁺ₜ₊₁) − (μ⁻ₜ − μ⁻ₜ₊₁), with μ = −dual. Write `price_from_ramp_duals(n, hour)`.

In [4]:
def price_from_ramp_duals(n, hour: int) -> float:
    """Coal's KKT price reconstruction at `hour` from PyPSA's ramp duals (n already solved)."""
    ### BEGIN SOLUTION
    up = -n.model.constraints["Generator-p-ramp_limit_up"].dual.sel(name="coal").fillna(0).to_numpy()
    dn = -n.model.constraints["Generator-p-ramp_limit_down"].dual.sel(name="coal").fillna(0).to_numpy()
    nxt = lambda a: a[hour + 1] if hour + 1 < len(a) else 0.0
    return float(PLANTS.loc["coal", "mc"] + (up[hour] - nxt(up)) - (dn[hour] - nxt(dn)))
    ### END SOLUTION

In [5]:
n = network()
n.optimize(solver_name="highs", log_to_console=False, progress=False, assign_all_duals=True)
assert np.isclose(price_from_ramp_duals(n, 16), n.buses_t.marginal_price["HR"].iloc[16]), "65 EUR/MWh, as in lesson 3"
print(f"16:00: price {n.buses_t.marginal_price['HR'].iloc[16]:.2f} = reconstruction {price_from_ramp_duals(n, 16):.2f} EUR/MWh")

16:00: price 65.00 = reconstruction 65.00 EUR/MWh


### Core

**3. A CO₂ budget with `extra_functionality`**

Write `co2_budget(limit_t)` that adds Σ emission factor × generation ≤ `limit_t` to PyPSA's model, solves, and returns a dict with the cost, the emissions and the carbon price (positive, €/t). Use a budget of 16,000 t for the day. Then check it twice: by finite differences, and against PyPSA's built-in `GlobalConstraint` (Maths-2) on `network(with_carriers=True)`.

In [6]:
def co2_budget(limit_t: float) -> dict:
    """Cost (EUR), emissions (t) and carbon price (EUR/t) with a daily CO2 budget added by extra_functionality."""
    ### BEGIN SOLUTION
    factors = pd.Series(PLANTS.co2)
    def add(n, snapshots):
        p = n.model.variables["Generator-p"].sel(name=list(factors.index))
        weights = pd.Series(factors.to_numpy(), index=pd.Index(factors.index, name="name")).to_xarray()
        n.model.add_constraints((p * weights).sum() <= limit_t, name="co2_budget")
    n = network()
    n.optimize(solver_name="highs", log_to_console=False, progress=False, extra_functionality=add)
    emissions = float((n.generators_t.p[factors.index] * factors).sum().sum())
    return {"cost": n.objective, "emissions": emissions,
            "carbon_price": -float(n.model.constraints["co2_budget"].dual.item()) + 0.0}
    ### END SOLUTION

In [7]:
unlimited = co2_budget(1e9)
capped = co2_budget(16_000)
assert unlimited["carbon_price"] == 0 and unlimited["emissions"] > 16_000, "without a binding budget, carbon is free"
assert np.isclose(capped["emissions"], 16_000) and capped["carbon_price"] > 0
fd = (co2_budget(15_999)["cost"] - co2_budget(16_001)["cost"]) / 2
assert np.isclose(fd, capped["carbon_price"], rtol=1e-6), "the dual is the cost saved per extra tonne"

gc = network(with_carriers=True)
gc.add("GlobalConstraint", "co2", type="primary_energy", carrier_attribute="co2_emissions", sense="<=", constant=16_000)
gc.optimize(solver_name="highs", log_to_console=False, progress=False)
assert np.isclose(gc.objective, capped["cost"]) and np.isclose(abs(gc.global_constraints.mu["co2"]), capped["carbon_price"])
print(f"unlimited emissions {unlimited['emissions']:,.0f} t | budget 16,000 t: carbon price {capped['carbon_price']:.2f} EUR/t "
      f"(finite difference {fd:.2f}; GlobalConstraint {abs(gc.global_constraints.mu['co2']):.2f}), "
      f"cost +{capped['cost'] - unlimited['cost']:,.0f} EUR")

unlimited emissions 18,254 t | budget 16,000 t: carbon price 63.64 EUR/t (finite difference 63.64; GlobalConstraint 63.64), cost +130,892 EUR


**4. Interpretation**

Your own constraint and PyPSA's `GlobalConstraint` give the same cost and carbon price. When would you still write the constraint yourself? And why does the carbon price here differ from Maths-2's 54.55 €/t, even though the emission factors are similar?

<!-- BEGIN SOLUTION -->
`GlobalConstraint` covers the standard case — a cap on emissions from carriers with an emission attribute — and is the right choice when it fits: less code, and other people reading the model recognise it. You write the constraint yourself when the rule does not fit that template: a budget that applies only to some plants or some hours, an emissions intensity target (t/MWh rather than t), a limit that combines several quantities (for example emissions and imports together), or anything with time-varying coefficients. `extra_functionality` gives you the full Linopy model, so any linear rule is possible — at the price of writing and testing it yourself, which is exactly what checking it against `GlobalConstraint` and finite differences does.

The carbon price differs because the plant data differ, not because the rule does. At the margin, cutting a tonne means replacing coal by gas, and the carbon price settles where the two cost the same including carbon — Maths-2's equal-marginal condition: c = (95 − 60) / (0.90 − 0.35) = 63.64 €/t. In Maths-2 gas cost 90 €/MWh, giving (90 − 60) / (0.90 − 0.35) = 54.55 €/t. At 63.64 €/t, coal and gas are equally expensive including carbon: 60 + 0.90 × 63.64 = 95 + 0.35 × 63.64 = 117.27 €/MWh. The ramp limits and the battery decide *in which hours* the switch happens and how much it costs in total, but the price of the last tonne is set by this one trade-off.
<!-- END SOLUTION -->

### Challenge

**5. The same budget in Pyomo**

Use the setup's `pyomo_dispatch(co2_limit=...)` to solve the day with the 16,000 t budget. Write `pyomo_budget(limit_t)` returning the cost and carbon price (positive, €/t, from the Pyomo `Suffix`), and show that Pyomo, your PyPSA constraint and `GlobalConstraint` agree. Then compare the **hourly prices** with and without the budget: in which hours does the budget raise the price, and by how much?

In [8]:
def pyomo_budget(limit_t: float) -> dict:
    """Cost (EUR), carbon price (EUR/t) and hourly prices (EUR/MWh) from Pyomo with a daily CO2 budget."""
    ### BEGIN SOLUTION
    m = pyomo_dispatch(co2_limit=limit_t)
    pyo.SolverFactory("appsi_highs").solve(m)
    return {"cost": pyo.value(m.cost), "carbon_price": -m.dual[m.co2] + 0.0,
            "prices": np.array([m.dual[m.balance[t]] for t in range(T)]) + 0.0}
    ### END SOLUTION

In [9]:
pb = pyomo_budget(16_000)
assert np.isclose(pb["cost"], capped["cost"], rtol=1e-9) and np.isclose(pb["carbon_price"], capped["carbon_price"], rtol=1e-6)
base = pyomo_dispatch()
pyo.SolverFactory("appsi_highs").solve(base)
base_prices = np.array([base.dual[base.balance[t]] for t in range(T)]) + 0.0
uplift = pd.Series(pb["prices"] - base_prices, index=HOURS, name="price_uplift_EUR_MWh")
assert (uplift >= -1e-6).all(), "a CO2 budget never lowers a price in this day"
print(f"Pyomo carbon price {pb['carbon_price']:.2f} EUR/t = PyPSA {capped['carbon_price']:.2f} EUR/t")
pd.DataFrame({"without_budget": base_prices, "with_budget": pb["prices"], "uplift": uplift}).round(2).T

Pyomo carbon price 63.64 EUR/t = PyPSA 63.64 EUR/t


                    0       1       2       3       4       5       6   \
without_budget   95.00   95.00   95.00   95.00   95.00   95.00   95.00   
with_budget     117.27  117.27  117.27  117.27  117.27  117.27  124.88   
uplift           22.27   22.27   22.27   22.27   22.27   22.27   29.88   

                    7       8       9   ...      14      15      16      17  \
without_budget   95.00   95.00   95.00  ...    0.00    0.00   65.00   95.00   
with_budget     117.27  117.27  117.27  ...  112.71  112.71  117.27  112.71   
uplift           22.27   22.27   22.27  ...  112.71  112.71   52.27   17.71   

                    18      19      20      21      22      23  
without_budget   95.00   95.00   95.00   95.00   95.00   95.00  
with_budget     121.84  124.88  124.88  117.27  117.27  117.27  
uplift           26.84   29.88   29.88   22.27   22.27   22.27  

[3 rows x 24 columns]

**6. Interpretation**

Look at the hourly price uplift. Explain the night-time prices with the budget exactly (117.27 €/MWh). Why do the midday hours rise the most — from 0 to over 110 €/MWh — even though no fossil plant is cheaper there?

<!-- BEGIN SOLUTION -->
With a binding budget, every MWh from a fossil plant carries an extra cost of (carbon price × its emission factor). At night both coal and gas run between their limits, so both satisfy the stationarity condition and the price equals either one's full cost: 60 + 0.90 × 63.64 = 95 + 0.35 × 63.64 = 117.27 €/MWh — an uplift of 22.27 €/MWh over the gas-set 95 €/MWh. Some evening hours rise a little more (to 121.84 or 124.88 €/MWh) because ramp and battery limits bind there too, as in lesson 3. The midday hours change most. Without the budget a little solar was curtailed and the marginal MWh at midday was worthless (price 0). With the budget the model squeezes out every tonne it can: coal ramps down further (its midday minimum falls from 379 to 205 MW), the battery stores solar for the evening, and no solar is curtailed any more. Now one extra MWh of free solar at midday would let coal produce one MWh less on its constrained ramp path — saving coal's cost *and* its carbon — so it is worth close to coal's full cost including carbon (117.27 €/MWh), slightly less because of the ramp and storage coupling (112.71 €/MWh). A carbon budget raises the value of clean energy most exactly in the hours where it used to be worth nothing.
<!-- END SOLUTION -->